<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/ProcessingLLMwithBatches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
device = 'cuda'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map = device, torch_dtype = torch.float16)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [17]:
import random

texts = [""]

questions_pool = [
    "is 1+1 equal to 2",
    "do human have eyes",
    "does dog have 6 legs"
]

template = (
    "Task: Answer the question using ONLY 'Yes' or 'No'. Do not output any other word.\n\n"
    "Text: {}\n"
    "Question: {}\n"
    "Answer (Yes/No):"
)

questions = [
    template.format(text, random.choice(questions_pool))
    for text in texts
    for _ in range(1024)
]

In [18]:
len(questions)

1024

In [19]:
%%time
answers = []
from tqdm import tqdm
batch_size = 512
for i in tqdm(range(0, len(questions), batch_size)):
  batch = questions[i:i+batch_size]
  inputs = tokenizer(batch, padding = True, truncation = True, return_tensors = 'pt')
  input_length = inputs.input_ids.shape[1]
  with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=5)
  generated_tokens = outputs[:, input_length:]
  decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

  answers.extend(decoded)

100%|██████████| 2/2 [01:42<00:00, 51.30s/it]

CPU times: user 1min 19s, sys: 22.6 s, total: 1min 41s
Wall time: 1min 42s


In [20]:
len(answers)

1024

In [21]:
answers

[' The text provides information about',
 ' No, dog has four',
 ' The text provides information about',
 ' Yes\nYou are an',
 ' The text provides information about',
 ' No, dog has ',
 ' The text provides sufficient information',
 ' Yes No The statement "',
 ' Yes\nYou are an',
 ' The text provides information about',
 ' Yes\n\nThe text provided',
 ' No, dog has ',
 ' Yes\nYou are an',
 ' Yes\n\nThe given text',
 ' The text provides information about',
 ' No, dog has ',
 ' Yes Yes',
 ' The text provides sufficient information',
 ' The text provides information about',
 ' The text provides information about',
 ' No, dog has ',
 ' The text provides information about',
 ' The text provides information about',
 ' The text provides information about',
 ' The text provides sufficient information',
 ' The text provides information about',
 ' No, dog has ',
 ' Yes\n\nThe given text',
 ' No, dog has ',
 ' The text provides sufficient information',
 ' The text provides sufficient information',
 